# 03 - Final irradiance-correction model (XGBoost)

## 1. Summary

This notebook trains the solar-irradiance prediction model.

The goal is to model the complex atmospheric effects (cloud cover, aerosols, water vapour) that the render engine does not capture on its own. An **XGBoost** model estimates a **multiplicative correction factor** \(k\) such that:

`irr_real = k * irr_sim`

with

`k = irr_real / (irr_sim + e)`

The `e` term avoids division by zero when irr_sim ≈ 0 and improves stability.

The model is trained on `dataset_master_tfg.csv`, which combines:
- Geometric and solar-visibility variables from the Unreal Engine simulation, including the normalized virtual irradiance.
- External atmospheric variables from Solcast.
- Real irradiance measured by the pyranometers.

Three days (`2025-04-11`, `2025-04-20`, `2025-10-07`) are held out as the test set. Each represents an atmospheric regime (overcast, variable, clear). `2025-10-07` is of particular interest because of the shadows cast on sensor P1.

The output is the trained model, saved as `model_final.pkl`, together with its configuration and performance metadata.

## 2. Training configuration

### 2.1 Environment and imports

In [ ]:
import os, json, hashlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import warnings

from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error

warnings.simplefilter(action='ignore', category=FutureWarning)

### 2.2 Training configuration

In [ ]:
CFG_FINAL = {
    "iteration_id": "final_model",
    "data_path": "../data/dataset_master_tfg.csv",
    "description": 
        """
    The best configuration for the minimal model.
        """,
    
    # Target definition
    "target": "k_eff",
    "k_clip": (0.0, 1.5),
    "sim_clip_lower": 1.0,
    "eps": 10.0,
    
    # Split
    "test_dates": ["2025-04-11", "2025-04-20", "2025-10-07"],
    "train_filter": {
       "min_altitude_deg": 5.0,
    },
    
    # Features
    "feature_sets": {
        "final": [
            # --- minimal ---
            "sim_irradiance_wm2",
            "cloud_opacity",

        ],
    },
    "feature_set_name": "final",
    
    # Model
    "model_params": dict(
        n_estimators=800,
        learning_rate=0.01,
        max_depth=4,
        min_child_weight=70,
        subsample=0.5,
        colsample_bytree=0.5,
        reg_alpha=0.1,
        reg_lambda=1.0,
        gamma=0.1,
        objective="reg:squarederror",
        eval_metric="rmse",
        n_jobs=-1,
        random_state=42,
    ),
    
    # Outputs
    "out_dir" : "../artifacts",
    "out_iterations": "../artifacts/iterations",
    "out_predictions": "../artifacts/predictions",
    "out_final": "../artifacts/final_model",
    
}

## 3. Loading and preparing the data

In [ ]:
def load_base_df(path):
    df = pd.read_csv(path)
    
    # Timestamp index
    df["timestamp"] = pd.to_datetime(df["Unnamed: 0"])
    df = df.set_index("timestamp").drop(columns=["Unnamed: 0"])
    
    # Bool->int 
    for c in [c for c in df.columns if df[c].dtype == bool]:
        df[c] = df[c].astype(int)
    df["date"] = df["date"].astype(str)
    
    return df

In [ ]:
def split_train_test(df, cfg):
    test_df = df[df["date"].isin(cfg["test_dates"])].copy()
    train_df = df[~df["date"].isin(cfg["test_dates"])].copy()

    # Train filtering by solar altitude
    min_alt = cfg["train_filter"].get("min_altitude_deg", 0)
    if "sim_altitude" in train_df.columns:
        train_df = train_df[train_df["sim_altitude"] > min_alt].copy()
    
    # Clean NaNs 
    train_df = train_df.dropna(subset=["real_irradiance", "sim_irradiance_wm2", "cloud_opacity"])
    
    return train_df, test_df

In [ ]:
def fingerprint_config(cfg):
    # Hash to identify iterations
    raw = json.dumps(cfg, sort_keys=True).encode("utf-8")
    return hashlib.md5(raw).hexdigest()[:10]

## 4. Building the feature vector

In [ ]:
def add_features(df, cfg):
    df = df.copy()
    df = df.sort_index()

    # --- CUSTOM FEATURES --- 
    
    # diffuse fraction
    sim_diffuse_wm2 = 0.01305 * df["sim_comp_amb_lux"]
    df["sim_diffuse_fraction"] = sim_diffuse_wm2 / (df['sim_irradiance_wm2'] + 1.0)
    df["sim_diffuse_fraction"] = df["sim_diffuse_fraction"].clip(0.0, 1.0)
    df["sim_diffuse_fraction"] = df["sim_diffuse_fraction"].fillna(0.0)
   
    # Temporal window
    s = df.groupby(df.index)["cloud_opacity"].first().sort_index()
    
    window = "60T"
    window_10 = 5   # 5 * 2min = 10 min
    window_20 = 10  # 10 * 2min = 20 min
    s_mean_10 = s.rolling(window_10, min_periods=1).mean()
    s_std_10  = s.rolling(window_10, min_periods=1).std().fillna(0)

    s_mean_20 = s.rolling(window_20, min_periods=1).mean()
    s_std_20  = s.rolling(window_20, min_periods=1).std().fillna(0)
    
    s_mean_60 = s.rolling(window, min_periods=1).mean()
    s_std_60  = s.rolling(window, min_periods=1).std().fillna(0)
    s_diff = s.diff().fillna(0)

    s_lag_1 = s.shift(1)
    s_lag_2 = s.shift(2)

    # --- Deltas ---
    s_delta_1 = s - s_lag_1
    s_delta_2 = s - s_lag_2
    
    for series in [s_lag_1, s_lag_2, s_delta_1, s_delta_2]:
        series.fillna(0, inplace=True)
    
    # Map back
    df["cloud_mean_60m"] = df.index.map(s_mean_60)
    df["cloud_std_60m"]  = df.index.map(s_std_60)

    df["cloud_mean_10m"] = df.index.map(s_mean_10)
    df["cloud_std_10m"]  = df.index.map(s_std_10)

    df["cloud_mean_20m"] = df.index.map(s_mean_20)
    df["cloud_std_20m"]  = df.index.map(s_std_20)

    df["cloud_opacity_lag_2m"] = df.index.map(s_lag_1)
    df["cloud_opacity_lag_4m"] = df.index.map(s_lag_2)

    df["cloud_delta_2m"] = df.index.map(s_delta_1)
    df["cloud_delta_4m"] = df.index.map(s_delta_2)

    # --------
    
    # Temporal interpolation of missing data (4 min max)
    df = df.interpolate(method="time", limit=2).ffill()

    # Dummies sensor
    df = pd.get_dummies(df, columns=["sensor"], prefix="sensor")
    return df

In [ ]:
def build_feature_list(df, cfg):
    base = list(cfg["feature_sets"][cfg["feature_set_name"]])

    # Automatically add sensor_* columns
    sensor_cols = sorted([c for c in df.columns if c.startswith("sensor_")])
    feats = base #+ sensor_cols --> SENSORS NOT ADDED

    # Dedupe
    seen = set()
    feats = [f for f in feats if not (f in seen or seen.add(f))]
    return feats

## 5. Training the model

In [ ]:
def build_k_eff(df, cfg):
    eps = float(cfg["eps"])
    sim = df["sim_irradiance_wm2"].astype(float).values
    real = df["real_irradiance"].astype(float).values

    k = real / (sim + eps)

    lo, hi = cfg["k_clip"]
    k = np.clip(k, lo, hi)

    mask = sim >= float(cfg["sim_clip_lower"])
    return k, mask

In [ ]:
def train_model(X_train, y_train, cfg):
    model = XGBRegressor(**cfg["model_params"])
    model.fit(X_train, y_train)
    return model

In [ ]:
def predict_irradiance(model, df, X, cfg):
    # pred_k * sim_irradiance 
    sim = df["sim_irradiance_wm2"].astype(float).values
    k_pred = model.predict(X)

    lo, hi = cfg["k_clip"]
    k_pred = np.clip(k_pred, lo, hi)

    pred_irr = np.maximum(k_pred * sim, 0.0)
    return k_pred, pred_irr

In [ ]:
def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    return {
        "r2": float(r2_score(y_true, y_pred)),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": rmse,
        "mbe": float(np.mean(y_pred - y_true)),
    }

In [ ]:
def extract_feature_importance(model, features, importance_type="gain"):
    """
    Extracts and returns feature importance as a sorted DataFrame.
    """
    booster = model.get_booster()
    score = booster.get_score(importance_type=importance_type)

    # Ensure all features appear
    data = []
    for f in features:
        data.append({
            "feature": f,
            "importance": float(score.get(f, 0.0))
        })

    df_imp = pd.DataFrame(data)
    df_imp = df_imp.sort_values("importance", ascending=False).reset_index(drop=True)
    return df_imp


def normalize_importance(df_importance):
    """
    Adds normalized importance (%) to a feature importance DataFrame.
    """
    df = df_importance.copy()
    total = df["importance"].sum()

    if total > 0:
        df["importance_pct"] = 100.0 * df["importance"] / total
    else:
        df["importance_pct"] = 0.0

    return df


def extract_all_feature_importances(model, features):
    """
    Extracts gain, weight and cover importances and returns them as dicts.
    """
    out = {}

    for itype in ["gain", "weight", "cover"]:
        df_imp = extract_feature_importance(
            model,
            features,
            importance_type=itype
        )

        # Only gain is normalized
        if itype == "gain":
            df_imp = normalize_importance(df_imp)

        out[itype] = df_imp.to_dict(orient="records")

    return out

In [ ]:
def run_experiment(cfg):
    os.makedirs(cfg["out_iterations"], exist_ok=True)
    os.makedirs(cfg["out_final"],      exist_ok=True)

    df0 = load_base_df(cfg["data_path"])
    train_raw, test_raw = split_train_test(df0, cfg)

    train_df = add_features(train_raw, cfg)
    test_df  = add_features(test_raw, cfg)
    k_train, mask = build_k_eff(train_df, cfg)
    train_df = train_df.iloc[np.where(mask)[0]].copy()
    k_train = k_train[mask]

    features = build_feature_list(train_df, cfg)
    X_train = train_df[features]
    X_test  = test_df[features]

    y_train = k_train
    
    y_test = test_df["real_irradiance"].values

    model = train_model(X_train, y_train, cfg)
    _, pred_irr = predict_irradiance(model, test_df, X_test, cfg)
    
    # Feature importance
    feature_importances = extract_all_feature_importances(model, features)

    # Global metrics
    m_global = compute_metrics(y_test, pred_irr)

    # Per-day metrics (aggregated)
    rows_day = []
    for d in cfg["test_dates"]:
        dd = test_df[test_df["date"] == d]
        if len(dd) == 0:
            continue
        _, irr_d = predict_irradiance(model, dd, dd[features], cfg)
        rows_day.append({"date": d, **compute_metrics(dd["real_irradiance"].values, irr_d)})

    # Artefacts & metadata
    exp_id = f"{cfg['iteration_id']}_{fingerprint_config(cfg)}"
    meta = {
        "exp_id": exp_id,
        "cfg": cfg,
        "n_train": int(len(train_df)),
        "n_test": int(len(test_df)),
        "features": features,
        "metrics_global": m_global,
        "metrics_by_day": rows_day,
        "feature_importance": feature_importances,
    }
    
    # Save metadata
    meta_path = os.path.join(cfg["out_final"], cfg["iteration_id"] + "_meta.json")
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2, ensure_ascii=False)

    # Save model
    import joblib
    model_path = os.path.join(cfg["out_final"], "final_model.pkl")
    joblib.dump(model, model_path)

    return model, meta, train_df, test_df

model, meta, train_df, test_df = run_experiment(CFG_FINAL)

In [ ]:
def _infer_sensor_name(df: pd.DataFrame) -> pd.Series:
    """
    Devuelve una serie con el nombre del sensor a partir de:
    - columna 'sensor' o 'sensor_name' si existe
    - dummies sensor_P0, sensor_P1, ... si existen
    """
    if "sensor" in df.columns:
        return df["sensor"].astype(str)
    if "sensor_name" in df.columns:
        return df["sensor_name"].astype(str)

    sensor_dummy_cols = [c for c in df.columns if c.startswith("sensor_")]
    if sensor_dummy_cols:
        
        idx = df[sensor_dummy_cols].values.argmax(axis=1)
        names = pd.Series([sensor_dummy_cols[i].replace("sensor_", "") for i in idx], index=df.index)

        all_zero = (df[sensor_dummy_cols].sum(axis=1) == 0)
        names[all_zero] = "unknown"
        return names

    # Fallback
    return pd.Series(["unknown"] * len(df), index=df.index)


def export_final_predictions(
    test_df: pd.DataFrame,
    model,
    cfg: dict,
    days: list,
    out_dir: str,
    model_label: str = "Final_sim_ML",
):
    os.makedirs(out_dir, exist_ok=True)

    features = build_feature_list(test_df, cfg)

    for day in days:
        dd = test_df[test_df["date"] == day].copy()
        if dd.empty:
            print(f"[WARN] No data for {day}")
            continue

        _, pred = predict_irradiance(model, dd, dd[features], cfg)

        out = pd.DataFrame(index=dd.index)
        out["real_irradiance"] = dd["real_irradiance"].values
        out["pred"] = pred
        out["sensor_name"] = _infer_sensor_name(dd)

        fname = f"preds_{model_label}_{day}.csv"
        fpath = os.path.join(out_dir, fname)
        out.to_csv(fpath)
        print(f"[OK] Exported {fpath} | n={len(out)}")


EXPORT_DAYS = ["2025-10-07", "2025-04-20", "2025-04-11"]

export_final_predictions(
    test_df=test_df,
    model=model,
    cfg=CFG_FINAL,
    days=EXPORT_DAYS,
    out_dir=CFG_FINAL["out_predictions"],
    model_label="Final_sim_ML",
)

## 6. Model evaluation

### 6.1 Global metrics

In [ ]:
global_metrics = pd.DataFrame(
    meta["metrics_global"],
    index=["Final model"]
)

global_metrics = global_metrics[["r2", "rmse", "mae", "mbe"]]

display(global_metrics.style
    .format({
        "r2": "{:.3f}",
        "rmse": "{:.2f}",
        "mae": "{:.2f}",
        "mbe": "{:.2f}",
    })
    .set_caption(f"Global summary – final model")
)

### 6.2 Per-day metrics

In [ ]:
df_by_day = (
    pd.DataFrame(meta["metrics_by_day"])
    .sort_values("date")
    .set_index("date")
)

display(df_by_day.style
    .format({
        "r2": "{:.3f}",
        "rmse": "{:.1f}",
        "mae": "{:.1f}",
        "mbe": "{:.1f}",
    })
    .set_caption("Model performance per test day")
)

## 7. Feature-importance analysis

The relative contribution of each input variable to the final model, to check its behaviour and consistency.

In [ ]:
def pretty_feature_name(f: str) -> str:
    if f.startswith("sensor_"):
        s = f.replace("sensor_", "")
        return f"Sensor {s}"

    mapping = {
        "cloud_opacity": "Cloud opacity",
        "cloud_cover": "Cloud cover",
        "sim_irradiance_wm2": "Simulated & normalized\nirradiance [W/m2]",
        "diffuse_fraction": "Diffuse fraction",
    }
    if f in mapping:
        return mapping[f]

    # fallback: snake_case -> "Title case"
    return f.replace("_", " ").strip().capitalize()

def plot_feature_importance(df_importance, top_k=10):
    """
    Plots normalized feature importance (gain %) for the top_k features.
    Expects columns: feature, importance_pct
    """
    dfp = (
        df_importance
        .sort_values("importance_pct", ascending=False)
        .head(top_k)
        .iloc[::-1]
        .copy()
    )

    dfp["feature_label"] = dfp["feature"].astype(str).apply(pretty_feature_name)

    fig, ax = plt.subplots(figsize=(7.6, 4.2))  

    ax.barh(dfp["feature_label"], dfp["importance_pct"])

    ax.set_xlabel("Relative importance (%)")
    ax.set_title("Feature importance (XGBoost gain)")

    ax.grid(axis="x", alpha=0.25)
    ax.grid(axis="y", visible=False)

    ax.xaxis.set_major_locator(mticker.MultipleLocator(10))

    xmax = float(dfp["importance_pct"].max()) if len(dfp) else 1.0
    ax.set_xlim(0, xmax * 1.12)

    offset = xmax * 0.015
    for y, v in enumerate(dfp["importance_pct"].tolist()):
        ax.text(v + offset, y, f"{v:.1f}%", va="center", fontsize=9)

    plt.tight_layout()
    plt.show()
    
df_imp = pd.DataFrame(meta["feature_importance"]["gain"])
plot_feature_importance(df_imp, top_k=8)

## 8. Result visualisation

### 8.1 Representative time series

Two representative examples of the final model's behaviour for the same sensor (P1): a clear day and a day with high atmospheric variability.

In [ ]:
def plot_day_sensor(test_df, model, cfg, day, sensor_name):
    features = build_feature_list(test_df, cfg)
    dd = test_df[test_df["date"] == day].copy()

    # Filter by sensor
    col = f"sensor_{sensor_name}"
    if col in dd.columns:
        dd = dd[dd[col] == 1]

    if dd.empty:
        print("No data for", day, sensor_name)
        return

    _, pred = predict_irradiance(model, dd, dd[features], cfg)
    y = dd["real_irradiance"].values
    m = compute_metrics(y, pred)

    # Plot
    plt.figure(figsize=(14,4))
    plt.plot(dd.index, y, color="black", linewidth=1.8, label="Irradiancia real")
    plt.plot(dd.index, pred, color="red", linestyle="--",linewidth=2.0, label="Irradiancia estimada")
    plt.title(f"{day} | {sensor_name} | R2={m['r2']:.3f} RMSE={m['rmse']:.1f}")
    plt.xlabel("Time (Local)")
    plt.ylabel("Irradiance [W/m²]")
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    plt.gca().xaxis.set_major_locator(mdates.HourLocator(interval=1))
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_day_sensor(test_df, model, CFG_FINAL, day="2025-10-07", sensor_name="P1")
plot_day_sensor(test_df, model, CFG_FINAL, day="2025-04-20", sensor_name="P1")
plot_day_sensor(test_df, model, CFG_FINAL, day="2025-04-11", sensor_name="P1")